# 06 — Leakage-safe categorical-pair target encoding

16個のカテゴリ特徴量から全120ペアを作り、各組み合わせの解約率を数値特徴量へ変換します。
目的変数を使うため、通常の全体集計ではリークします。このNotebookではouter CVとinner CVを
入れ子にして、各行のラベルがその行のencodingへ入らないようにします。

```text
outer train: inner 5-fold OOF encoding
outer valid: outer train全体だけで作ったmapping
test       : outer train全体だけで作ったmapping
```

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from config import Baseline
from features import (
    build_nested_pair_te_for_outer_fold,
    get_base_features,
    get_categorical_pairs,
)

TARGET = Baseline.TARGET
ID_COLUMN = Baseline.ID_COLUMN
FOLD_COLUMN = Baseline.FOLD_COLUMN

train = pd.read_csv(ROOT / "input" / "train.csv")
test = pd.read_csv(ROOT / "input" / "test.csv")
train[TARGET] = train[TARGET].map({"Yes": 1, "No": 0})
folds = pd.read_csv(ROOT / "output" / "stkfolds.csv")
train = train.merge(folds, on=ID_COLUMN, how="left", validate="one_to_one")
assert train[[TARGET, FOLD_COLUMN]].notna().all().all()
print("train:", train.shape, "test:", test.shape)

## カテゴリペア

数値dtypeでも低カーディナリティの `SeniorCitizen` はカテゴリとして扱います。

In [ ]:
BASE_FEATURES, BASE_NUM, BASE_CAT = get_base_features(
    train, TARGET, ID_COLUMN, min_nunique=Baseline.MIN_NUMERIC_UNIQUE
)
PAIR_COLUMNS = get_categorical_pairs(BASE_CAT)
assert len(PAIR_COLUMNS) == len(BASE_CAT) * (len(BASE_CAT) - 1) // 2
print("numeric:", BASE_NUM)
print("categorical columns:", len(BASE_CAT))
print("pair features:", len(PAIR_COLUMNS))
display(pd.DataFrame(PAIR_COLUMNS, columns=["feature_1", "feature_2"]).head(10))

## 全outer foldを一度だけ生成

元実装にあった「確認用に1 foldを生成した直後、同じfoldをもう一度生成する」処理は削除しました。
各ファイルにはIDを残し、学習時にIDで再整列して行ずれを検出できるようにします。

In [ ]:
FEATURE_DIR = ROOT / "output" / "nested_pair_te"
FEATURE_DIR.mkdir(parents=True, exist_ok=True)
pd.DataFrame(PAIR_COLUMNS, columns=["feature_1", "feature_2"]).to_csv(
    FEATURE_DIR / "pair_columns.csv", index=False
)

for outer_fold in sorted(train[FOLD_COLUMN].unique()):
    print("=" * 70)
    print("outer fold:", outer_fold)
    train_te, valid_te, test_te = build_nested_pair_te_for_outer_fold(
        train=train,
        test=test,
        target=TARGET,
        categorical_pairs=PAIR_COLUMNS,
        outer_fold=outer_fold,
        id_column=ID_COLUMN,
        fold_column=FOLD_COLUMN,
        n_inner_splits=Baseline.N_SPLITS,
        smoothing=20.0,
        random_state=Baseline.SEED,
    )

    train_mask = train[FOLD_COLUMN].ne(outer_fold)
    valid_mask = ~train_mask
    assert train_te.shape == (int(train_mask.sum()), len(PAIR_COLUMNS) + 1)
    assert valid_te.shape == (int(valid_mask.sum()), len(PAIR_COLUMNS) + 1)
    assert test_te.shape == (len(test), len(PAIR_COLUMNS) + 1)
    assert np.isfinite(train_te.drop(columns=ID_COLUMN).to_numpy()).all()
    assert np.isfinite(valid_te.drop(columns=ID_COLUMN).to_numpy()).all()
    assert np.isfinite(test_te.drop(columns=ID_COLUMN).to_numpy()).all()

    fold_tag = str(outer_fold).replace("/", "_")
    train_te.to_parquet(FEATURE_DIR / f"outer_fold_{fold_tag}_train.parquet", index=False)
    valid_te.to_parquet(FEATURE_DIR / f"outer_fold_{fold_tag}_valid.parquet", index=False)
    test_te.to_parquet(FEATURE_DIR / f"outer_fold_{fold_tag}_test.parquet", index=False)
    print("saved:", train_te.shape, valid_te.shape, test_te.shape)

## 注意点

120列は元のカテゴリ情報から作った強く相関する特徴量です。特徴量を増やせば必ずAUCが上がるわけではありません。
線形モデルには交互作用を直接渡せる利点がありますが、木モデルは元特徴量から同様の分割を学習できるため、
冗長性によって悪化することもあります。採用はNotebook 06-FE2のOOF差で決めます。